In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from art.estimators.classification import PyTorchClassifier

from sklearn.preprocessing import MinMaxScaler

import time
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parents[1]))

from art.attacks.evasion import FastGradientMethod
from utils.functions import run_simple_full_attack, test_simple_model, create_simple_dataset
from utils.models import SimpleNet

In [34]:
## Step 0: Define constants
data_file = "../../data/RandomPos_0709.csv"
bn_model_filename = "RandomPos-full"
adv_model_filename = "RandomPos-adv-train"
eps=0.04

In [ ]:
(x_train, y_train), (x_test, y_test) = create_simple_dataset(data_file=data_file, divide_by=100)

dataset length: 31600
% of attackers in training dataset: 0.2650316455696203
% of attackers in testing dataset: 0.20569620253164558


In [32]:
# Step 2: adversarial train model
# Run a simple adversarial ART attack, end to end. model_filename = None to not save the model
def adv_train(data_file, divide_by, bn_model_filename, adv_model_filename, Attack, **attack_kwargs):

    ## Step 2: Create the model
    model = SimpleNet()

    # Step 2a: Define the loss function and the optimizer

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    ## Step 3: Create the ART classifier

    classifier = PyTorchClassifier(
        model=model,
        clip_values=(x_test.min(), x_test.max()),
        loss=criterion,
        optimizer=optimizer,
        input_shape=(11,),
        nb_classes=2,
    )

    start = time.time()
    # Step 4: Load model
    model.load_state_dict(torch.load(f"../../saved_models/{bn_model_filename}.pth", weights_only=True))
    model.eval()

    # Step 6: Generate adversarial test examples
    attack = Attack(classifier, **attack_kwargs)

    x_train_adv = attack.generate(x=x_train)
    return x_train_adv



In [49]:
x_train_adv = adv_train(data_file=data_file,
          divide_by=2,
          bn_model_filename=bn_model_filename,
          adv_model_filename=adv_model_filename,
          Attack=FastGradientMethod,
          eps=eps)

In [50]:
print("Negative values in x_train:", np.sum(x_train_adv == 1))

Negative values in x_train: 0


In [51]:
x_train

array([[0.0000000e+00, 1.4752889e-03, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [3.1645578e-07, 1.4752889e-03, 0.0000000e+00, ..., 9.5868651e-03,
        8.3029540e-03, 1.7447094e-02],
       [6.3291156e-07, 1.4752889e-03, 0.0000000e+00, ..., 9.5385024e-03,
        8.7999050e-03, 9.1456268e-03],
       ...,
       [7.9990532e-03, 2.7046963e-03, 1.4507008e-02, ..., 8.1901299e-04,
        1.3209557e-02, 8.9747705e-02],
       [7.9993699e-03, 2.7046963e-03, 1.4507008e-02, ..., 8.1901299e-04,
        1.3209557e-02, 8.9747705e-02],
       [7.9996865e-03, 2.7046963e-03, 1.4507008e-02, ..., 8.1901299e-04,
        1.3209557e-02, 8.9747705e-02]], shape=(25280, 11), dtype=float32)

In [ ]:
index = 2
col_header = {
    1: "ReceiverID",
    2: "SenderID",
    6: "MssgCount"
}
train_unique = set(x_train[:, index])
adv_unique = set(x_train_adv[:, index])
new_values = adv_unique - train_unique

print(f"== Unique `{col_header[index]}` values ==")
print(f"x_train unique: \t\t {len(train_unique)}")
print(f"x_train_adv unique: \t\t {len(adv_unique)}")
print(f"New values in x_train_adv:\t {len(new_values)}")

adv_unique

== Unique `SenderID` values ==
x_train unique: 		 60
x_train_adv unique: 		 111
New values in x_train_adv:	 60


{np.float32(0.0),
 np.float32(0.00024588147),
 np.float32(0.00049176294),
 np.float32(0.0009835259),
 np.float32(0.0012294074),
 np.float32(0.0014752889),
 np.float32(0.0017211704),
 np.float32(0.0019670518),
 np.float32(0.0022129333),
 np.float32(0.0024588148),
 np.float32(0.0027046963),
 np.float32(0.0029505778),
 np.float32(0.0031964593),
 np.float32(0.0034423408),
 np.float32(0.0036882223),
 np.float32(0.0039341035),
 np.float32(0.0041799853),
 np.float32(0.0044258665),
 np.float32(0.0049176295),
 np.float32(0.0051635113),
 np.float32(0.0059011555),
 np.float32(0.0061470373),
 np.float32(0.0066388003),
 np.float32(0.0071305633),
 np.float32(0.0073764445),
 np.float32(0.0076223263),
 np.float32(0.007868207),
 np.float32(0.008114089),
 np.float32(0.0083599705),
 np.float32(0.008605852),
 np.float32(0.008851733),
 np.float32(0.009097615),
 np.float32(0.0093434965),
 np.float32(0.009589378),
 np.float32(0.009835259),
 np.float32(0.010081141),
 np.float32(0.0103270225),
 np.float32(0.01

: 